# Centauri Bloom | Contributor training
One model. Many contributors. Built from scratch.

This is a bounded single-worker research baseline, not a live distributed training service. Run only on hardware you are authorized to use, with applicable account permissions and an approved budget. No cloud resources are created and nothing is uploaded. For a first run, CPU is sufficient.

This is an offline contributor smoke test, not the canonical continuation pipeline. To train the latest published model, use colab-bloom-0.1-continue.ipynb or the dataset-specific continuation notebooks.

## 1. Prepare the repository
Review the source and use a maintainer-approved commit. Clone or attach the repository to your Jupyter environment first. Set `REPO` to the directory containing `pyproject.toml`. For Kaggle, import this notebook into an authorized session, attach a reviewed repository copy, and enable network only if needed for dependencies or the public corpus. Do not store account tokens in notebook cells.

In [ ]:
from pathlib import Path
import sys, subprocess, json
REPO = Path.cwd()
if not (REPO / 'pyproject.toml').is_file() and (REPO.parent / 'pyproject.toml').is_file():
    REPO = REPO.parent
assert (REPO / 'bloom' / 'train.py').is_file(), 'Set REPO to the reviewed Bloom repository'
subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO, check=True)

In [ ]:
# Review the dependency list before installing into this environment.
print((REPO / 'pyproject.toml').read_text())
# Uncomment only if needed and authorized:
# subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO)], check=True)

## 2. Select and verify data
Default: bundled original offline fixture. The reviewed historical corpus is larger and requires network for the pinned sources. The selected Hugging Face datasets and contributor instruction data are not yet part of this pipeline.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
USE_PUBLIC_CORPUS = False
WORK = REPO / 'runs' / 'notebook'
WORK.mkdir(parents=True, exist_ok=True)
SOURCE = REPO / ('datasets/public-domain-v1/sources.json' if USE_PUBLIC_CORPUS else 'tests/fixtures/text/sources.json')
DATA = WORK / ('public-domain' if USE_PUBLIC_CORPUS else 'fixture')
if not DATA.exists():
    subprocess.run([sys.executable, '-m', 'bloom.dataset', 'build', str(SOURCE), str(DATA)], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'bloom.dataset', 'verify', str(DATA)], cwd=REPO, check=True)
from bloom.notebook_workflow import preflight
preflight()


## 3. Train and save
Begin with the small CPU baseline. Increase the budget only on authorized hardware. The trainer refuses oversized runs and existing output paths. For weights-only continuation, set a reviewed parent path and its independently supplied digest. This does not resume AdamW state or create a validated collaborative round submission.

In [ ]:
STEPS = 100
BATCH = 8
DEVICE = 'cpu'  # or 'cuda' if available and authorized
SEED = 42
MAX_TOKENS = 100_000
OUTPUT = WORK / 'baseline-001'
PARENT = None
PARENT_SHA256 = None
cmd = [sys.executable, '-m', 'bloom.train', str(DATA), str(OUTPUT), '--steps', str(STEPS), '--batch-size', str(BATCH), '--device', DEVICE, '--seed', str(SEED), '--max-tokens', str(MAX_TOKENS)]
if PARENT is not None:
    assert PARENT_SHA256, 'Supply trusted parent digest'
    cmd += ['--parent', str(PARENT), '--parent-sha256', PARENT_SHA256]
subprocess.run(cmd, cwd=REPO, check=True)

## 4. Review and share results
Inspect the metrics and manifest. Keep the test split held out. Share only reviewed artifacts and non-sensitive results through the project issue workflow. Never share credentials, private datasets or executable checkpoints. The maintainer will coordinate independent worker assignments in issue #5; this notebook does not automatically submit or aggregate weights.

In [ ]:
import hashlib
metrics = json.loads((OUTPUT / 'metrics.json').read_text())
manifest = json.loads((OUTPUT / 'manifest.json').read_text())
assert hashlib.sha256((OUTPUT / 'model.safetensors').read_bytes()).hexdigest() == manifest['weights_sha256']
print(json.dumps(metrics, indent=2))
print('Artifact directory:', OUTPUT)
print('Dataset:', manifest['dataset_sha256'])
print('Weights:', manifest['weights_sha256'])